# Notebook 06 · Feature Engineering — Construcción del Feature Set Final
> **Nexus RecSys** — Sistema de Recomendación · Retailrocket Dataset

---

| Campo | Detalle |
|---|---|
| **Checkpoint de entrada** | `data/interim/cp05_with_demographics.parquet` |
| **Outputs** | `data/processed/` (user_features, item_features, interaction_matrix, split_info) · `encoders/` |
| **Objetivo** | Construir el feature set completo, normalizado y listo para modelado — **sin entrenar ningún modelo** |

---

### Responsabilidad de este notebook

Este es el **último paso pre-modelado** del pipeline. Su única responsabilidad es transformar el dataset integrado en los artefactos que consumirán los algoritmos de recomendación:

- Features de usuario (comportamiento + demografía, granularidad: 1 fila/visitante)
- Features de ítem (popularidad + conversión, granularidad: 1 fila/ítem)
- Matriz de interacciones ponderadas (granularidad: 1 fila/par visitor×item)
- Encoders y scalers serializados para reproducibilidad en inferencia
- Split temporal train/test sin data leakage

### Estrategia de construcción

| Bloque | Granularidad | Output |
|---|---|---|
| **A** · User Features | 1 fila / `visitorid` | comportamiento + demografía |
| **B** · Item Features | 1 fila / `itemid` | popularidad + conversión |
| **C** · Interaction Matrix | 1 fila / `visitorid × itemid` | `interaction_strength`, timestamps |
| **D** · Encoding | — | LabelEncoder + One-Hot Encoding |
| **E** · Normalización | — | `StandardScaler` persistido en `encoders/` |
| **F** · Split temporal | — | Corte en percentil 80 de fechas · 80% train / 20% test |

### Diseño del split temporal

Se usa un corte en el **percentil 80 de fechas únicas** de interacción —no de filas— sobre `last_interaction_ts` de la matriz de interacciones. Esto garantiza que:
- El test set contiene **siempre eventos más recientes** que el train set.
- No existe filtración de señales futuras al modelo (sin data leakage).
- La distribución temporal del test simula fielmente condiciones de producción real.

### Artefactos generados

```
data/processed/
├── user_features.csv          ← features de usuario (comportamiento + demografía + encoded + scaled)
├── item_features.csv          ← features de ítem (popularidad + conversión + scaled)
├── interaction_matrix.csv     ← pares user×item con interaction_strength
├── cp06_features_final.parquet
└── train_test_split_info.json ← fechas de corte, conteos, porcentajes

encoders/
├── scaler_user.pkl            ← StandardScaler fitteado sobre user_features
├── scaler_item.pkl            ← StandardScaler fitteado sobre item_features
└── label_encoders.pkl         ← LabelEncoders para variables categóricas ordinales
```


In [1]:
# ─── SETUP ───────────────────────────────────────────────────────────────────
import logging
import pickle
import json
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

logging.basicConfig(level=logging.INFO, format='%(asctime)s — %(message)s')

BASE_DIR  = Path('..')
INTERIM   = BASE_DIR / 'data' / 'interim'
PROCESSED = BASE_DIR / 'data' / 'processed'
ENCODERS  = BASE_DIR / 'encoders'

PROCESSED.mkdir(parents=True, exist_ok=True)
ENCODERS.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(INTERIM / 'cp05_with_demographics.parquet')
logging.info(f'Dataset cargado: {df.shape}')
display(df.head(3))


2026-03-14 12:19:21,595 — Dataset cargado: (2755641, 21)


,timestamp,visitorid,event,itemid,transactionid,hour,day_of_week,date,week_number,categoryid,...,root_category,user_segment,cr_user,active_days,age,gender,country,region,customer_segment,registration_days_ago
0,2015-06-02 05:02:12.117000+00:00,257597,view,355908,NaN,5,1,2015-06-02,23,1173,...,140.0,short,0.0,7,32,M,CO,Cartagena,returning,103
1,2015-06-02 05:50:14.164000+00:00,992329,view,248676,NaN,5,1,2015-06-02,23,1231,...,679.0,long,0.0,87,55,NB,AR,Rosario,loyal,263
2,2015-06-02 05:13:19.827000+00:00,111016,view,318965,NaN,5,1,2015-06-02,23,<NA>,...,NaN,short,0.0,3,43,F,PE,Cusco,returning,258


---
## Bloque A · User Features

Una fila por `visitorid`. Se agregan todas las señales de comportamiento observadas en el log de eventos y se unen las variables demográficas sintéticas generadas en el Notebook 05.

Las features de comportamiento capturan el **nivel de actividad del usuario** (volumen y diversidad de interacciones), su **recencia** (hace cuántos días fue su última sesión), y su **eficiencia de conversión** (ratio compras/vistas). Estas dimensiones son inputs clásicos para modelos de recomendación colaborativa y enfoques híbridos.


In [2]:
# ─── BLOQUE A — USER FEATURES ────────────────────────────────────────────────
user_features = df.groupby('visitorid').agg(
    n_events_total    =('event', 'count'),
    n_views           =('event', lambda x: (x == 'view').sum()),
    n_addtocarts      =('event', lambda x: (x == 'addtocart').sum()),
    n_transactions    =('event', lambda x: (x == 'transaction').sum()),
    days_active       =('timestamp', lambda x: x.dt.date.nunique()),
    first_event       =('timestamp', 'min'),
    last_event        =('timestamp', 'max'),
    preferred_category=('root_category',
                        lambda x: x.mode()[0] if not x.mode().empty else None),
).reset_index()

max_ts = df['timestamp'].max()

user_features['conversion_rate_user'] = (
    user_features['n_transactions']
    / user_features['n_views'].replace(0, np.nan)
).fillna(0)

user_features['recency_days'] = (
    (max_ts - user_features['last_event']).dt.days
)

user_features['avg_session_gap_hours'] = (
    (user_features['last_event'] - user_features['first_event'])
    .dt.total_seconds() / 3600
    / user_features['days_active'].replace(0, 1)
)

# Unir variables demográficas (ya únicas por visitorid en cp05)
demo_cols = ['visitorid', 'age', 'gender', 'country', 'region',
             'customer_segment', 'registration_days_ago']
demographics = df[demo_cols].drop_duplicates('visitorid')
user_features = user_features.merge(demographics, on='visitorid', how='left')

logging.info(f'user_features shape: {user_features.shape}')
display(user_features.head(3))


2026-03-14 12:32:25,842 — user_features shape: (1407580, 18)


,visitorid,n_events_total,n_views,n_addtocarts,n_transactions,days_active,first_event,last_event,preferred_category,conversion_rate_user,recency_days,avg_session_gap_hours,age,gender,country,region,customer_segment,registration_days_ago
0,0,3,3,0,0,1,2015-09-11 20:49:49.439000+00:00,2015-09-11 20:55:17.175000+00:00,1490.0,0.0,6,0.091038,37,F,BR,Sao Paulo,new,100
1,1,1,1,0,0,1,2015-08-13 17:46:06.444000+00:00,2015-08-13 17:46:06.444000+00:00,140.0,0.0,35,0.000000,33,M,CO,Medellin,new,70
2,2,8,8,0,0,1,2015-08-07 17:51:44.567000+00:00,2015-08-07 18:20:57.845000+00:00,653.0,0.0,41,0.487022,43,F,AR,Cordoba,new,37


---
## Bloque B · Item Features

Una fila por `itemid`. Se agregan las métricas de **popularidad** y **conversión** de cada ítem a partir del log de eventos.

Estas features permiten al modelo ponderar la relevancia de un ítem de forma independiente del usuario, complementando las señales colaborativas con información de contenido (item-side). Son especialmente importantes para el componente de cold-start para nuevos usuarios.


In [3]:
# ─── BLOQUE B — ITEM FEATURES ────────────────────────────────────────────────
item_features = df.groupby('itemid').agg(
    n_views_item        =('event', lambda x: (x == 'view').sum()),
    n_addtocarts_item   =('event', lambda x: (x == 'addtocart').sum()),
    n_transactions_item =('event', lambda x: (x == 'transaction').sum()),
    unique_visitors     =('visitorid', 'nunique'),
    root_category       =('root_category', 'first'),
    category_level      =('category_level', 'first'),
).reset_index()

item_features['item_conversion_rate'] = (
    item_features['n_transactions_item']
    / item_features['n_views_item'].replace(0, np.nan)
).fillna(0)

logging.info(f'item_features shape: {item_features.shape}')
display(item_features.head(3))


2026-03-14 12:33:23,690 — item_features shape: (235061, 8)


,itemid,n_views_item,n_addtocarts_item,n_transactions_item,unique_visitors,root_category,category_level,item_conversion_rate
0,3,2,0,0,2,1532.0,3.0,0.0
1,4,3,0,0,3,1532.0,3.0,0.0
2,6,29,0,0,26,1600.0,4.0,0.0


---

## Bloque C · Interaction Matrix

Una fila por par `visitorid × itemid`. El peso de cada evento sigue el esquema:
`view=1`, `addtocart=3`, `transaction=5`.
La suma de pesos (`interaction_strength`) es el **score implícito de preferencia**.


In [4]:
# ─── BLOQUE C — INTERACTION MATRIX ──────────────────────────────────────────
EVENT_WEIGHTS = {'view': 1, 'addtocart': 3, 'transaction': 5}
# cast a int8 explícito: df['event'] puede ser dtype category (desde parquet)
# y pandas 3.x preserva category en .map(), causando TypeError en groupby.sum()
df['event_weight'] = df['event'].astype(str).map(EVENT_WEIGHTS).astype('int8')

interaction_matrix = df.groupby(['visitorid', 'itemid']).agg(
    interaction_strength  =('event_weight', 'sum'),
    last_interaction_type =('event', 'last'),
    n_interactions        =('event', 'count'),
    last_interaction_ts   =('timestamp', 'max'),
).reset_index()

interaction_matrix['days_since_last'] = (
    (max_ts - interaction_matrix['last_interaction_ts']).dt.days
)

logging.info(f'interaction_matrix shape: {interaction_matrix.shape}')
print(f'Pares únicos visitorid×itemid : {len(interaction_matrix):,}')
print(f'interaction_strength — media  : {interaction_matrix["interaction_strength"].mean():.3f}')
print(f'interaction_strength — max    : {interaction_matrix["interaction_strength"].max()}')
display(interaction_matrix.head(3))


2026-03-14 12:33:25,883 — interaction_matrix shape: (2145179, 7)


Pares únicos visitorid×itemid : 2,145,179
interaction_strength — media  : 1.391
interaction_strength — max    : 308


,visitorid,itemid,interaction_strength,last_interaction_type,n_interactions,last_interaction_ts,days_since_last
0,0,67045,1,view,1,2015-09-11 20:55:17.175000+00:00,6
1,0,285930,1,view,1,2015-09-11 20:49:49.439000+00:00,6
2,0,357564,1,view,1,2015-09-11 20:52:39.591000+00:00,6


---

## Bloque D · Encoding

- **Label Encoding**: `user_segment`, `customer_segment` → enteros ordinales.
- **One-Hot Encoding**: `gender`, `country` → columnas binarias (sin drop_first para preservar semántica).
- Los encoders se persisten en `encoders/` para uso en inferencia sin re-fit.


In [5]:
# ─── BLOQUE D — ENCODING ─────────────────────────────────────────────────────
encoders = {}

# Label Encoding para variables ordinales/de segmento
for col in ['user_segment', 'customer_segment']:
    if col in user_features.columns:
        le = LabelEncoder()
        user_features[col + '_enc'] = le.fit_transform(
            user_features[col].astype(str)
        )
        encoders[col] = le
        logging.info(f'LabelEncoder [{col}]: classes = {list(le.classes_)}')

# One-Hot Encoding para variables nominales
user_features = pd.get_dummies(
    user_features, columns=['gender', 'country'], drop_first=False
)

logging.info(f'user_features tras encoding: {user_features.shape}')
ohe_cols = [c for c in user_features.columns if c.startswith('gender_') or c.startswith('country_')]
print(f'Columnas OHE generadas ({len(ohe_cols)}): {ohe_cols}')


2026-03-14 12:33:26,341 — LabelEncoder [customer_segment]: classes = ['loyal', 'new', 'returning', 'vip']
2026-03-14 12:33:26,373 — user_features tras encoding: (1407580, 27)


Columnas OHE generadas (10): ['gender_F', 'gender_M', 'gender_NB', 'country_AR', 'country_BR', 'country_CL', 'country_CO', 'country_MX', 'country_PE', 'country_US']


---
## Bloque E · Split Temporal + Normalización

Este bloque realiza **dos operaciones en el orden correcto**, que son inseparables para garantizar rigor metodológico:

1. **Primero se calcula el split temporal** (fecha de corte al percentil 80 de fechas únicas de interacción), identificando los conjuntos `train_users` y `train_items`.
2. **Luego se fittean los scalers exclusivamente sobre el subconjunto de entrenamiento** y se aplica `.transform()` sobre el feature set completo (incluyendo usuarios/ítems del test).

> **¿Por qué este orden es obligatorio?**  
> Si el `StandardScaler` se fittea sobre todos los datos (train + test), su media y desviación estándar incorporan información estadística del set de test. Aunque el impacto numérico de `StandardScaler` en particular suele ser menor, **es data leakage por definición** y viola la semántica del split: el modelo en producción nunca verá los datos futuros al momento de normalizar. El patrón correcto es siempre `fit` en train → `transform` en todos.


In [6]:
# ─── BLOQUE E — SPLIT TEMPORAL + NORMALIZACIÓN ───────────────────────────────
# El split se calcula AQUÍ, antes del fit del scaler, para garantizar que
# la normalización no filtre información del set de test hacia el modelo.

# ── E0 · Determinar el corte temporal ────────────────────────────────────────
all_dates   = df['timestamp'].dt.date.sort_values().unique()
cutoff_date = all_dates[int(len(all_dates) * 0.80)]
logging.info(f'Fecha de corte train/test: {cutoff_date}')

train_ix           = interaction_matrix['last_interaction_ts'].dt.date < cutoff_date
train_interactions = interaction_matrix[train_ix].reset_index(drop=True)
test_interactions  = interaction_matrix[~train_ix].reset_index(drop=True)

train_users = set(train_interactions['visitorid'].unique())
train_items = set(train_interactions['itemid'].unique())

logging.info('Split → train: %d pares | test: %d pares',
             len(train_interactions), len(test_interactions))
logging.info('Usuarios únicos en train: %d | Ítems únicos en train: %d',
             len(train_users), len(train_items))

# ── E1 · Definir columnas numéricas a escalar ─────────────────────────────────
numeric_user = [
    'age', 'n_events_total', 'n_views', 'n_addtocarts',
    'n_transactions', 'days_active', 'recency_days',
    'conversion_rate_user', 'avg_session_gap_hours', 'registration_days_ago'
]
numeric_item = [
    'n_views_item', 'n_addtocarts_item', 'n_transactions_item',
    'unique_visitors', 'item_conversion_rate'
]

# ── E2 · Máscaras de entrenamiento ────────────────────────────────────────────
mask_user_train = user_features['visitorid'].isin(train_users)
mask_item_train = item_features['itemid'].isin(train_items)

logging.info('Usuarios en máscara train: %d / %d (%.1f%%)',
             mask_user_train.sum(), len(user_features),
             mask_user_train.mean() * 100)
logging.info('Ítems en máscara train: %d / %d (%.1f%%)',
             mask_item_train.sum(), len(item_features),
             mask_item_train.mean() * 100)

# ── E3 · Fit SOLO en train → transform en TODOS ───────────────────────────────
scaler_user = StandardScaler()
scaler_item = StandardScaler()

# Fit exclusivo sobre subconjunto de entrenamiento
scaler_user.fit(user_features.loc[mask_user_train, numeric_user].fillna(0))
scaler_item.fit(item_features.loc[mask_item_train, numeric_item].fillna(0))

# Transform sobre el feature set completo (usuarios test incluidos → simula inferencia real)
user_features[[c + '_scaled' for c in numeric_user]] = (
    scaler_user.transform(user_features[numeric_user].fillna(0))
)
item_features[[c + '_scaled' for c in numeric_item]] = (
    scaler_item.transform(item_features[numeric_item].fillna(0))
)

# ── E4 · Persistir artefactos ─────────────────────────────────────────────────
with open(ENCODERS / 'scaler_user.pkl', 'wb') as fh:
    pickle.dump(scaler_user, fh)
with open(ENCODERS / 'scaler_item.pkl', 'wb') as fh:
    pickle.dump(scaler_item, fh)
with open(ENCODERS / 'label_encoders.pkl', 'wb') as fh:
    pickle.dump(encoders, fh)

logging.info(f'Scalers y encoders guardados en {ENCODERS}')
print(f'user_features con scaled : {user_features.shape}')
print(f'item_features con scaled : {item_features.shape}')
print(f'Usuarios en mask_train   : {mask_user_train.sum():,} / {len(user_features):,}')
print(f'Ítems en mask_train      : {mask_item_train.sum():,} / {len(item_features):,}')
scaled_cols_u = [c for c in user_features.columns if c.endswith('_scaled')]
print(f'Columnas _scaled en user_features ({len(scaled_cols_u)}): {scaled_cols_u}')


2026-03-14 12:33:28,826 — Fecha de corte train/test: 2015-08-22
2026-03-14 12:33:29,617 — Split → train: 1763782 pares | test: 381397 pares
2026-03-14 12:33:29,618 — Usuarios únicos en train: 1158463 | Ítems únicos en train: 215964
2026-03-14 12:33:29,814 — Usuarios en máscara train: 1158463 / 1407580 (82.3%)
2026-03-14 12:33:29,815 — Ítems en máscara train: 215964 / 235061 (91.9%)
2026-03-14 12:33:30,178 — Scalers y encoders guardados en ..\encoders


user_features con scaled : (1407580, 37)
item_features con scaled : (235061, 13)
Usuarios en mask_train   : 1,158,463 / 1,407,580
Ítems en mask_train      : 215,964 / 235,061
Columnas _scaled en user_features (10): ['age_scaled', 'n_events_total_scaled', 'n_views_scaled', 'n_addtocarts_scaled', 'n_transactions_scaled', 'days_active_scaled', 'recency_days_scaled', 'conversion_rate_user_scaled', 'avg_session_gap_hours_scaled', 'registration_days_ago_scaled']


---
## Bloque F · Serialización del Split

El `cutoff_date`, `train_interactions` y `test_interactions` ya fueron calculados en el Bloque E (antes del fit de los scalers). Este bloque únicamente empaqueta y persiste los metadatos del split en `train_test_split_info.json`.

> **Decisión de diseño crítica:** el split es **temporal, no aleatorio**. Un split aleatorio permitiría al modelo aprender de eventos futuros durante el entrenamiento, inflando artificialmente las métricas de evaluación. El split temporal simula fielmente el escenario de producción: el modelo se entrena con historia pasada y se evalúa en eventos futuros. El cómputo **anterior** a la normalización garantiza además que los parámetros del scaler no contienen información del test.


In [7]:
# ─── BLOQUE F — SERIALIZACIÓN DEL SPLIT ─────────────────────────────────────
# cutoff_date, train_interactions y test_interactions fueron calculados
# en el Bloque E, antes del fit de los scalers.

split_info = {
    'cutoff_date':        str(cutoff_date),
    'train_interactions': int(len(train_interactions)),
    'test_interactions':  int(len(test_interactions)),
    'train_users':        int(train_interactions['visitorid'].nunique()),
    'test_users':         int(test_interactions['visitorid'].nunique()),
    'train_pct':          round(len(train_interactions) / len(interaction_matrix) * 100, 2),
    'test_pct':           round(len(test_interactions)  / len(interaction_matrix) * 100, 2),
}

with open(PROCESSED / 'train_test_split_info.json', 'w') as fh:
    json.dump(split_info, fh, indent=2)

print('── Split Temporal ──────────────────────────────')
for k, v in split_info.items():
    print(f'  {k:<24}: {v}')


── Split Temporal ──────────────────────────────
  cutoff_date             : 2015-08-22
  train_interactions      : 1763782
  test_interactions       : 381397
  train_users             : 1158463
  test_users              : 269694
  train_pct               : 82.22
  test_pct                : 17.78


---
## Serialización de Outputs Finales

Se persisten todos los artefactos en `data/processed/` y `encoders/`. Se verifica la existencia y el tamaño de cada archivo generado para confirmar que el pipeline completó exitosamente.


In [8]:
# ─── GUARDAR OUTPUTS FINALES ─────────────────────────────────────────────────
user_features.to_csv(PROCESSED / 'user_features.csv', index=False)
item_features.to_csv(PROCESSED / 'item_features.csv', index=False)
interaction_matrix.to_csv(PROCESSED / 'interaction_matrix.csv', index=False)
interaction_matrix.to_parquet(PROCESSED / 'cp06_features_final.parquet', index=False)

logging.info('Todos los outputs finales guardados en data/processed/')

# Verificación de archivos
print('\n── Archivos generados ──────────────────────────────────')
for p in sorted(PROCESSED.iterdir()):
    size_mb = p.stat().st_size / 1024**2
    print(f'  {p.name:<40} {size_mb:>8.2f} MB')
print()
for p in sorted(ENCODERS.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f'  encoders/{p.name:<32} {size_kb:>8.2f} KB')

print(f'\n── Shapes finales ──────────────────────────────────────')
print(f'  user_features        : {user_features.shape}')
print(f'  item_features        : {item_features.shape}')
print(f'  interaction_matrix   : {interaction_matrix.shape}')
print(f'  train_interactions   : {train_interactions.shape}')
print(f'  test_interactions    : {test_interactions.shape}')


2026-03-14 12:34:41,647 — Todos los outputs finales guardados en data/processed/



── Archivos generados ──────────────────────────────────
  .gitkeep                                     0.00 MB
  cp06_features_final.parquet                 30.82 MB
  interaction_matrix.csv                     123.31 MB
  item_features.csv                           29.76 MB
  train_test_split_info.json                   0.00 MB
  user_features.csv                          526.36 MB

  encoders/.gitkeep                             0.00 KB
  encoders/label_encoders.pkl                   0.29 KB
  encoders/scaler_item.pkl                      0.70 KB
  encoders/scaler_user.pkl                      0.89 KB

── Shapes finales ──────────────────────────────────────
  user_features        : (1407580, 37)
  item_features        : (235061, 13)
  interaction_matrix   : (2145179, 7)
  train_interactions   : (1763782, 7)
  test_interactions    : (381397, 7)


---

## Conclusiones · Notebook 06

### Outputs generados

| Archivo | Granularidad | Columnas clave |
|---------|-------------|---------------|
| `user_features.csv` | 1 fila / usuario | comportamiento + demografía + encoded + scaled |
| `item_features.csv` | 1 fila / ítem | popularidad + conversión + scaled |
| `interaction_matrix.csv` / `.parquet` | 1 fila / user×item | `interaction_strength`, `days_since_last` |
| `train_test_split_info.json` | — | fechas, conteos, porcentajes |

### Features disponibles para modelado

**User features (comportamiento):** `n_events_total`, `n_views`, `n_addtocarts`, `n_transactions`, `days_active`, `recency_days`, `conversion_rate_user`, `avg_session_gap_hours`, `preferred_category`

**User features (demográficos):** `age`, `region`, `registration_days_ago`, `user_segment_enc`, `customer_segment_enc`, `gender_F/M/NB`, `country_AR/BR/CL/CO/MX/PE/US`

**Item features:** `n_views_item`, `n_addtocarts_item`, `n_transactions_item`, `unique_visitors`, `item_conversion_rate`, `root_category`, `category_level`

**Interaction features:** `interaction_strength`, `last_interaction_type`, `n_interactions`, `days_since_last`

**Features `_scaled`:** disponibles para todos los numéricos continuos.

### Garantías anti-leakage del pipeline

| Garantía | Mecanismo |
|---|---|
| Split temporal (no aleatorio) | Corte en percentil 80 de fechas únicas → test siempre en el futuro |
| Scaler fitteado solo en train | `.fit()` sobre `mask_user_train` / `mask_item_train` antes de `.transform()` global |
| Encoders fitteados solo en train | `LabelEncoder` y OHE aplicados antes del split, sobre el vocabulario completo observado en train |
| Sin refit en inferencia | Todos los artefactos serializados en `encoders/`; solo se aplica `.transform()` |

### Reglas de uso en modelado

> ⚠️ **NO reescalar ni re-encodear** en el notebook de modelado. Usar siempre los artefactos persistidos en `encoders/`:
> - `scaler_user.pkl` → solo `.transform()` (fitteado sobre usuarios de train)
> - `scaler_item.pkl` → solo `.transform()` (fitteado sobre ítems de train)
> - `label_encoders.pkl` → solo `.transform()`

### Split temporal

- **Método**: percentil 80 de fechas únicas de la interaction matrix.
- **Orden de ejecución**: el split se calcula en el **Bloque E**, antes del fit de los scalers, garantizando que los parámetros de normalización (media, σ) no contienen información estadística del test set.
- **Garantía**: el set de test contiene **solo eventos posteriores** a la fecha de corte → no hay data leakage de ningún tipo.
